In [46]:
"""
Day 2 - Sales Analytics Without Pandas
Squad Python Specialization - Data Science, AI & Analytics

Tasks 1-7 use only the standard library. pandas is allowed in task 8, to check your work.
"""


import csv
import math
from pathlib import Path

DATA_PATH = Path("data") / "electronics_sales.csv"
OUTPUT_PATH = Path("output") / "sales_report.txt"

In [47]:
import os
print("Working directory:", os.getcwd())
print("Files here:", os.listdir("."))
print("Data folder exists:", os.path.isdir("data"))
if os.path.isdir("data"):
    print("Inside data:", os.listdir("data"))

Working directory: c:\Users\Josephine Mackylah\Desktop\day2-sales-analytics
Files here: ['.venv', 'data', 'expected_report.txt', 'requirements.txt', 'sales_analysis.ipynb']
Data folder exists: True
Inside data: ['electronics_sales.csv']


In [48]:
def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV with the csv module and return the raw rows."""
    try:
        with path.open(newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            rows = list(reader)
    except FileNotFoundError:
        print(f"File not found: {path}")
        return []
    print(f"Rows read: {len(rows)}")
    print(f"Columns found: {', '.join(reader.fieldnames or [])}")
    return rows


raw_rows = read_sales()

Rows read: 50
Columns found: sale_id, sale_date, store, product, quantity, unit_price_kes, payment_method


In [49]:
def parse_number(value: str | None) -> float | None:
    """Convert a messy numeric string to a float, or return None if invalid."""
    if value is None:
        return None

    cleaned = value.strip().upper().replace("KES", "").strip()
    cleaned = cleaned.replace(",", "")

    if cleaned in ("", "N/A", "NONE", "NULL", "-"):
        return None

    word_to_num = {
        "ZERO": 0, "ONE": 1, "TWO": 2, "THREE": 3, "FOUR": 4, "FIVE": 5,
        "SIX": 6, "SEVEN": 7, "EIGHT": 8, "NINE": 9, "TEN": 10,
    }
    if cleaned in word_to_num:
        return float(word_to_num[cleaned])

    try:
        number = float(cleaned)
    except ValueError:
        return None

    return number if math.isfinite(number) else None


print(parse_number("two"))               # 2.0
print(parse_number("  KES 1,200.50  "))  # 1200.5
print(parse_number("    N/A"))           # None
print(parse_number("nan"))               # None

2.0
1200.5
None
None


In [50]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    """Return clean rows and skipped rows with reasons."""
    clean_rows = []
    skipped_rows = []

    def find_value(row, *names):
        normalized = {
            key.lower().replace("_", "").replace(" ", ""): value
            for key, value in row.items()
        }
        for name in names:
            target = name.lower().replace("_", "").replace(" ", "")
            if target in normalized:
                return normalized[target]
            for key, value in normalized.items():
                if target in key:
                    return value
        return None

    for row_number, row in enumerate(rows, start=1):
        quantity = parse_number(find_value(row, "quantity", "qty"))
        unit_price = parse_number(find_value(row, "unit_price", "unit price", "price"))

        if quantity is None:
            skipped_rows.append({"row": row_number, "reason": "Invalid quantity", "data": row})
            continue
        if unit_price is None:
            skipped_rows.append({"row": row_number, "reason": "Invalid unit price", "data": row})
            continue

        clean_row = dict(row)
        clean_row["quantity"] = quantity
        clean_row["unit_price"] = unit_price
        clean_row["revenue"] = quantity * unit_price   # quantity x unit price, as the brief says
        clean_rows.append(clean_row)

    return clean_rows, skipped_rows


clean_rows, skipped_rows = convert_rows(raw_rows)
print(len(clean_rows), len(skipped_rows))   # these should add up to 50
for item in skipped_rows:
    print(item["row"], item["reason"])

47 3
30 Invalid unit price
35 Invalid quantity
39 Invalid unit price


In [51]:
def total_revenue(rows: list[dict]) -> float:
    """Calculate total revenue as quantity × unit price."""
    total = 0.0

    for row in rows:
        quantity = row.get("quantity")
        unit_price = row.get("unit_price", row.get("unit_price_kes"))

        if not isinstance(quantity, (int, float)):
            quantity = parse_number(str(quantity))
        if not isinstance(unit_price, (int, float)):
            unit_price = parse_number(str(unit_price))

        if quantity is not None and unit_price is not None:
            total += quantity * unit_price

    return total


print(f"Total Revenue: KES {total_revenue(convert_rows(read_sales())[0]):,.2f}")

Rows read: 50
Columns found: sale_id, sale_date, store, product, quantity, unit_price_kes, payment_method
Total Revenue: KES 1,317,210.00


In [52]:
def revenue_by(rows: list[dict], field: str) -> dict[str, float]:
    """Total revenue grouped by `field`, using a plain dict.

    Rows with a missing or blank `field` are skipped.
    """
    totals = {}

    for row in rows:
        key = row.get(field)
        if key is None or str(key).strip() == "":
            continue

        # row["revenue"] fails loudly if convert_rows forgot to add it,
        # instead of silently adding 0.0 like row.get("revenue", 0.0) did
        totals[key] = totals.get(key, 0.0) + row["revenue"]

    return totals


def print_breakdown(title: str, totals: dict[str, float]) -> None:
    """Print totals highest first; ties are listed A to Z."""
    print(title)
    for name, revenue in sorted(totals.items(), key=lambda item: (-item[1], item[0])):
        print(f"  {name}: KES {revenue:,.2f}")


# Testing and calling the function
product_totals = revenue_by(clean_rows, "product")
store_totals = revenue_by(clean_rows, "store")
payment_totals = revenue_by(clean_rows, "payment_method")

print_breakdown("Product revenue breakdown:", product_totals)
print_breakdown("Store revenue breakdown:", store_totals)
print_breakdown("Payment method breakdown:", payment_totals)

# Sanity check: each breakdown should add up to the same grand total
print(round(sum(product_totals.values()), 2) == round(total_revenue(clean_rows), 2))

Product revenue breakdown:
  HP 15 Laptop: KES 389,994.00
  Hisense 43in Smart TV: KES 272,993.00
  Lenovo Tab M10: KES 199,992.00
  Tecno Spark 20: KES 161,988.00
  Infinix Hot 40: KES 63,996.00
  Samsung Galaxy A15: KES 56,997.00
  JBL Go 3 Speaker: KES 49,491.00
  Anker Power Bank 20000: KES 38,691.00
  Xiaomi Redmi Buds: KES 31,990.00
  Oraimo FreePods: KES 19,593.00
  USB-C Charger 25W: KES 19,487.00
  Safaricom Neon Ray: KES 11,998.00
Store revenue breakdown:
  Kisumu Mega Plaza: KES 449,678.00
  Nairobi CBD: KES 337,483.00
  Westgate: KES 235,979.00
  Mombasa City Mall: KES 154,082.00
  Nakuru Westside: KES 139,988.00
Payment method breakdown:
  M-Pesa: KES 738,568.00
  Cash: KES 310,773.00
  Card: KES 267,869.00
True


In [53]:
print([name for name in dir() if not name.startswith("_")])

['DATA_PATH', 'HIGH_VALUE_THRESHOLD', 'In', 'OUTPUT_PATH', 'Out', 'Path', 'THRESHOLD_KES', 'buffer', 'clean_rows', 'convert_rows', 'count', 'csv', 'exit', 'get_ipython', 'high_value', 'high_value_sales', 'idx', 'io', 'item', 'math', 'mean', 'median', 'minimum_maximum', 'open', 'os', 'parse_number', 'path', 'payment_totals', 'print_breakdown', 'print_report', 'product', 'product_totals', 'quit', 'rank', 'raw_rows', 'read_sales', 'redirect_stdout', 'rev_max', 'rev_mean', 'rev_median', 'rev_min', 'revenue', 'revenue_by', 'revenue_max', 'revenue_mean', 'revenue_median', 'revenue_min', 'revenue_stream', 'row', 'sale', 'skipped_details', 'skipped_rows', 'sorted_products', 'store', 'store_high_value_counts', 'store_totals', 'top_n', 'top_three', 'total', 'total_rev', 'total_revenue', 'verify_with_pandas']


In [54]:
def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    """Return the n highest (key, value) pairs, highest first."""
    # Multi-level sort:
    # 1. Primary: Sort by revenue descending (negative float values for reverse sorting)
    # 2. Secondary: Sort by product name alphabetically (A-Z) to break ties cleanly
    sorted_items = sorted(
        totals.items(), key=lambda item: (-item[1], item[0].lower())
    )

    return sorted_items[:n]


# --- Verification with Product Dataset ---
# product_totals was calculated from the previous step
top_three = top_n(product_totals, n=3)

print("TOP 3 HIGHEST REVENUE PRODUCTS")
for rank, (product, revenue) in enumerate(top_three, start=1):
    print(f"{rank}. {product} — KES {revenue:,.2f}")



TOP 3 HIGHEST REVENUE PRODUCTS
1. HP 15 Laptop — KES 389,994.00
2. Hisense 43in Smart TV — KES 272,993.00
3. Lenovo Tab M10 — KES 199,992.00


In [55]:
def high_value(rows: list[dict], threshold: float) -> list[dict]:
    """Transactions whose revenue is at or above threshold (use a comprehension)."""
    return [row for row in rows if row.get("revenue", 0.0) >= threshold]


# --- Process High-Value Metrics ---
# Assuming 'clean_rows' comes from your salvaged convert_rows(read_sales()) pipeline
THRESHOLD_KES = 50000.0

# 1. Filter using the list comprehension
high_value_sales = high_value(clean_rows, THRESHOLD_KES)

print(
    f"--- HIGH-VALUE TRANSACTIONS (>= KES {THRESHOLD_KES:,.2f}) ---"
)
for sale in high_value_sales:
    print(
        f"ID: {sale['sale_id']} | Store: {sale['store']} | Product: {sale['product']} | Revenue: KES {sale['revenue']:,.2f}"
    )

# 2. Count occurrences per store using a dictionary comprehension
# Extracts unique store names from the filtered transactions, then counts their frequency
store_high_value_counts = {
    store: sum(1 for sale in high_value_sales if sale["store"] == store)
    for store in set(sale["store"] for sale in high_value_sales)
}

print("\n--- HIGH-VALUE TRANSACTION COUNTS BY STORE ---")
for store, count in sorted(
    store_high_value_counts.items(), key=lambda x: x, reverse=True
):
    print(f"Store: {store}: {count} transaction(s)")

--- HIGH-VALUE TRANSACTIONS (>= KES 50,000.00) ---
ID: S-3013 | Store: Westgate | Product: Lenovo Tab M10 | Revenue: KES 99,996.00
ID: S-3017 | Store: Kisumu Mega Plaza | Product: Hisense 43in Smart TV | Revenue: KES 155,996.00
ID: S-3022 | Store: Nairobi CBD | Product: HP 15 Laptop | Revenue: KES 259,996.00
ID: S-3036 | Store: Nakuru Westside | Product: Tecno Spark 20 | Revenue: KES 53,996.00
ID: S-3048 | Store: Kisumu Mega Plaza | Product: HP 15 Laptop | Revenue: KES 129,998.00

--- HIGH-VALUE TRANSACTION COUNTS BY STORE ---
Store: Westgate: 1 transaction(s)
Store: Nakuru Westside: 1 transaction(s)
Store: Nairobi CBD: 1 transaction(s)
Store: Kisumu Mega Plaza: 2 transaction(s)


In [56]:
# Reusable statistics - implement these yourself, no statistics module.
def mean(values: list[float]) -> float:
    """Calculate the arithmetic mean of a list of numbers."""
    if not values:
        raise ValueError("The list of values is empty.")
    return sum(values) / len(values)



def median(values: list[float]) -> float:
    """Calculate the median value of a list of numbers."""
    if not values:
        raise ValueError("The list of values is empty.")

    
    sorted_values = sorted(values)
    n = len(sorted_values)
    mid_index = n // 2

    if n % 2 == 0:  # Even number of elements
        return (sorted_values[mid_index - 1] + sorted_values[mid_index]) / 2
    else:  # Odd number of elements
        return sorted_values[mid_index]



def minimum_maximum(values: list[float]) -> tuple[float, float]:
    """Find the minimum and maximum values without using min() or max() built-ins."""

    if not values:
        raise ValueError("The list of values is empty.")

    min_value = float('inf')
    max_value = float('-inf')

    for value in values:
        if value < min_value:
            min_value = value
        if value > max_value:
            max_value = value

    return min_value, max_value

# mean([1, 2, 3, 4, 5, 6, 7, 8, 9])  # Returns: 5.5
# median([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])  # Returns: 5.5
# minimum_maximum([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])  # Returns: (1, 9)

# Assuming 'clean_rows' comes from your salvaged convert_rows(read_sales()) pipeline
# Extract revenue array using a list comprehension
revenue_stream = [row["revenue"] for row in clean_rows]

# Compute metrics
revenue_mean = mean(revenue_stream)
revenue_median = median(revenue_stream)
revenue_min, revenue_max = minimum_maximum(revenue_stream)


# Execution Printout
print("--- PER-TRANSACTION REVENUE STATISTICS ---")
print(f"Mean Transaction Value   : KES {revenue_mean:,.2f}")
print(f"Median Transaction Value : KES {revenue_median:,.2f}")
print(f"Minimum Transaction Value: KES {revenue_min:,.2f}")
print(f"Maximum Transaction Value: KES {revenue_max:,.2f}")


--- PER-TRANSACTION REVENUE STATISTICS ---
Mean Transaction Value   : KES 28,025.74
Median Transaction Value : KES 10,998.00
Minimum Transaction Value: KES 1,499.00
Maximum Transaction Value: KES 259,996.00


In [57]:
def verify_with_pandas(path: Path, my_results: dict) -> None:
    """Recompute the headline numbers with pandas and print MATCH or MISMATCH for each."""
    try:
        import pandas as pd          # imported here so steps 1-7 still run without pandas
    except ImportError:
        print("pandas is not installed; skipping verification.")
        return

    # Read everything as text so pandas does not guess types for the messy columns
    df = pd.read_csv(path, dtype=str)

    word_to_num = {
        "ZERO": 0.0, "ONE": 1.0, "TWO": 2.0, "THREE": 3.0, "FOUR": 4.0, "FIVE": 5.0,
        "SIX": 6.0, "SEVEN": 7.0, "EIGHT": 8.0, "NINE": 9.0, "TEN": 10.0,
    }

    def clean_number(val):
        """Same rules as parse_number: strip KES and commas, convert number words,
        and reject blanks, N/A, nan and inf."""
        if pd.isna(val):
            return float("nan")
        cleaned = str(val).strip().upper().replace("KES", "").replace(",", "").strip()
        if cleaned in ("", "N/A", "NONE", "NULL", "-"):
            return float("nan")
        if cleaned in word_to_num:
            return word_to_num[cleaned]
        try:
            number = float(cleaned)
        except ValueError:
            return float("nan")
        return number if math.isfinite(number) else float("nan")

    price_col = [col for col in df.columns if "price" in col.lower()][0]
    qty_col = [col for col in df.columns if "quantity" in col.lower() or col.lower() == "qty"][0]

    df["qty_clean"] = df[qty_col].apply(clean_number)
    df["price_clean"] = df[price_col].apply(clean_number)

    # Drop the rows my own convert_rows would skip
    df_clean = df.dropna(subset=["qty_clean", "price_clean"]).copy()
    df_clean["revenue"] = df_clean["qty_clean"] * df_clean["price_clean"]

    pandas_results = {
        "total_revenue": float(df_clean["revenue"].sum()),
        "mean": float(df_clean["revenue"].mean()),
        "median": float(df_clean["revenue"].median()),
        "min": float(df_clean["revenue"].min()),
        "max": float(df_clean["revenue"].max()),
    }

    print("--- PANDAS VERIFICATION ---")
    for metric, pd_val in pandas_results.items():
        my_val = my_results.get(metric)
        if my_val is None:
            status = "MISMATCH (metric missing from my results)"
        elif math.isclose(pd_val, my_val, abs_tol=0.01):
            status = "MATCH"
        else:
            status = f"MISMATCH (pandas {pd_val:,.2f} vs mine {my_val:,.2f})"
        print(f"{metric.upper().replace('_', ' '):<15}: {status}")
        
# Veify the custom calculations against pandas results
verify_with_pandas(
    DATA_PATH,
    {
        "total_revenue": total_revenue(clean_rows),
        "mean": revenue_mean,
        "median": revenue_median,
        "min": revenue_min,
        "max": revenue_max,
    },
)

--- PANDAS VERIFICATION ---
TOTAL REVENUE  : MATCH
MEAN           : MATCH
MEDIAN         : MATCH
MIN            : MATCH
MAX            : MATCH


In [58]:
if __name__ == "__main__":
    # Ingest data records
    path = DATA_PATH if "DATA_PATH" in globals() else Path("sales_data.csv")
    raw_rows = read_sales(path)

    # Clean and parse rows (including the salvaged 'two' quantity row)
    clean_rows, skipped_rows = convert_rows(raw_rows)

    # Compute custom statistical aggregates
    revenue_stream = [row["revenue"] for row in clean_rows]
    total_rev = sum(revenue_stream)
    product_totals = revenue_by(clean_rows, "product")
    top_three = top_n(product_totals, n=3)
    high_value_sales = high_value(clean_rows, threshold=50000.0)

    rev_mean = mean(revenue_stream)
    rev_median = median(revenue_stream)
    rev_min, rev_max = minimum_maximum(revenue_stream)

    # --- Print Sales Report ---
    print(f"=== Sales report: September 2026 ===")
    print(f"Rows read          : {len(raw_rows)}")
    print(f"Rows used          : {len(clean_rows)}")

    skipped_details = ", ".join(
        [f"{s['data'].get('sale_id', 'Unknown')}: {s['reason']}" for s in skipped_rows]
    )
    print(f"Rows skipped       : {len(skipped_rows)}   ({skipped_details})")
    print(f"\nTotal revenue (KES): {total_rev:,.2f}")

    print(f"\nRevenue by product (KES)")
    sorted_products = sorted(
        product_totals.items(), key=lambda x: x[1], reverse=True
    )
    for product, total in sorted_products:
        print(f"  {product:<25} {total:,.2f}")

    print(f"\nTop 3 products")
    for idx, (product, total) in enumerate(top_three, start=1):
        print(f"  {idx}. {product} - KES {total:,.2f}")

    print(f"\nHigh-value transactions (>= KES 50,000.00): {len(high_value_sales)}")
    for sale in high_value_sales:
        print(
            f"  {sale['sale_id']}  {sale['sale_date']}  {sale['store']:<18}  {sale['product']:<22}  KES {sale['revenue']:,.2f}"
        )

    print(f"\nTransaction revenue statistics")
    print(f"  mean   : {rev_mean:,.2f}")
    print(f"  median : {rev_median:,.2f}")
    print(f"  min    : {rev_min:,.2f}")
    print(f"  max    : {rev_max:,.2f}\n")

    # --- Pandas Automated Verification Workflow ---
    df = pd.read_csv(path)

    # Mirroring parsing transformations inside pandas DataFrame
    word_to_num = {
        "ONE": 1.0,
        "TWO": 2.0,
        "THREE": 3.0,
        "FOUR": 4.0,
        "FIVE": 5.0,
        "SIX": 6.0,
        "SEVEN": 7.0,
        "EIGHT": 8.0,
        "NINE": 9.0,
        "TEN": 10.0,
    }

    df["qty_clean"] = df["quantity"].apply(
        lambda v: word_to_num.get(str(v).strip().upper())
        if str(v).strip().upper() in word_to_num
        else pd.to_numeric(str(v).strip(), errors="coerce")
    )
    df["price_clean"] = df["unit_price_kes"].apply(
        lambda v: pd.to_numeric(
            str(v).strip().upper().replace("KES", "").replace(",", ""),
            errors="coerce",
        )
    )

    df_clean = df.dropna(subset=["qty_clean", "price_clean"]).copy()
    df_clean["revenue"] = df_clean["qty_clean"] * df_clean["price_clean"]

    pd_total_rev = float(df_clean["revenue"].sum())
    pd_prod_rev = df_clean.groupby("product")["revenue"].sum().to_dict()
    pd_top_3 = sorted(
        pd_prod_rev.items(), key=lambda item: (-item[1], item[0].lower())
    )[:3]
    pd_mean = float(df_clean["revenue"].mean())
    pd_median = float(df_clean["revenue"].median())

    # --- Print Verification Output ---
    print("=== pandas verification ===")

    # Total Revenue check
    tot_status = (
        "MATCH"
        if np.isclose(total_rev, pd_total_rev)
        else f"MISMATCH (pure={total_rev:,.2f}, pandas={pd_total_rev:,.2f})"
    )
    print(f"Total revenue      : {tot_status}")

    # Revenue by Product check
    prod_status = (
        "MATCH"
        if all(
            np.isclose(product_totals[k], pd_prod_rev.get(k, 0.0))
            for k in product_totals
        )
        else "MISMATCH"
    )
    print(f"Revenue by product : {prod_status}")

    # Top 3 check
    top3_status = "MATCH" if top_three == pd_top_3 else "MISMATCH"
    print(f"Top 3              : {top3_status}")

    # Mean and Median checks combined
    stats_status = (
        "MATCH"
        if np.isclose(rev_mean, pd_mean) and np.isclose(rev_median, pd_median)
        else f"MISMATCH (pure_mean={rev_mean:,.2f}, pandas_mean={pd_mean:,.2f} | pure_median={rev_median:,.2f}, pandas_median={pd_median:,.2f})"
    )
    print(f"Mean / median      : {stats_status}")


Rows read: 50
Columns found: sale_id, sale_date, store, product, quantity, unit_price_kes, payment_method
=== Sales report: September 2026 ===
Rows read          : 50
Rows used          : 47
Rows skipped       : 3   (S-3030: Invalid unit price, S-3035: Invalid quantity, S-3039: Invalid unit price)

Total revenue (KES): 1,317,210.00

Revenue by product (KES)
  HP 15 Laptop              389,994.00
  Hisense 43in Smart TV     272,993.00
  Lenovo Tab M10            199,992.00
  Tecno Spark 20            161,988.00
  Infinix Hot 40            63,996.00
  Samsung Galaxy A15        56,997.00
  JBL Go 3 Speaker          49,491.00
  Anker Power Bank 20000    38,691.00
  Xiaomi Redmi Buds         31,990.00
  Oraimo FreePods           19,593.00
  USB-C Charger 25W         19,487.00
  Safaricom Neon Ray        11,998.00

Top 3 products
  1. HP 15 Laptop - KES 389,994.00
  2. Hisense 43in Smart TV - KES 272,993.00
  3. Lenovo Tab M10 - KES 199,992.00

High-value transactions (>= KES 50,000.00): 5
 

NameError: name 'pd' is not defined

In [ ]:
import csv
import io
import math
from contextlib import redirect_stdout
from pathlib import Path

DATA_PATH = Path("data") / "electronics_sales.csv"
OUTPUT_PATH = Path("output") / "sales_report.txt"
HIGH_VALUE_THRESHOLD = 50000.0

In [ ]:
  print(f"High-value transactions (>= KES {HIGH_VALUE_THRESHOLD:,.2f}): {len(high_value_sales)}")

High-value transactions (>= KES 50,000.00): 5


In [66]:


def main():
    # Ingest and clean
    raw_rows = read_sales(DATA_PATH)
    clean_rows, skipped_rows = convert_rows(raw_rows)

    # Compute aggregates
    revenue_stream = [row["revenue"] for row in clean_rows]
    total_rev = sum(revenue_stream)
    product_totals = revenue_by(clean_rows, "product")
    top_three = top_n(product_totals, n=3)
    high_value_sales = high_value(clean_rows, HIGH_VALUE_THRESHOLD)

    stores = {row["store"] for row in clean_rows}
    high_by_store = {
        store: sum(1 for sale in high_value_sales if sale["store"] == store)
        for store in stores
    }

    rev_mean = mean(revenue_stream)
    rev_median = median(revenue_stream)
    rev_min, rev_max = minimum_maximum(revenue_stream)

    buffer = io.StringIO()
    with redirect_stdout(buffer):
        # --- Sales report ---
        print("=== Sales report: September 2026 ===")
        print(f"Rows read          : {len(raw_rows)}")
        print(f"Rows used          : {len(clean_rows)}")

        skipped_details = ", ".join(
            f"{s['data'].get('sale_id', 'Unknown')}: {s['reason']}" for s in skipped_rows
        )
        print(f"Rows skipped       : {len(skipped_rows)}   ({skipped_details})")
        print(f"\nTotal revenue (KES): {total_rev:,.2f}")

        print("\nRevenue by product (KES)")
        for product, total in sorted(product_totals.items(), key=lambda x: (-x[1], x[0])):
            print(f"  {product:<25} {total:,.2f}")

        print("\nTop 3 products")
        for idx, (product, total) in enumerate(top_three, start=1):
            print(f"  {idx}. {product} - KES {total:,.2f}")

        print(f"\nHigh-value transactions (>= KES {HIGH_VALUE_THRESHOLD:,.2f}): {len(high_value_sales)}")
        for sale in high_value_sales:
            print(
                f"  {sale['sale_id']}  {sale['sale_date']}  {sale['store']:<18}  "
                f"{sale['product']:<22}  KES {sale['revenue']:,.2f}"
            )

        print("\nHigh-value sales per store")
        for store, count in sorted(high_by_store.items(), key=lambda x: (-x[1], x[0])):
            print(f"  {store:<18} {count}")

        print("\nTransaction revenue statistics")
        print(f"  mean   : {rev_mean:,.2f}")
        print(f"  median : {rev_median:,.2f}")
        print(f"  min    : {rev_min:,.2f}")
        print(f"  max    : {rev_max:,.2f}\n")

        # --- pandas verification (pandas imported only here) ---
        try:
            import pandas as pd
        except ImportError:
            print("pandas is not installed; skipping verification.")
        else:
            df = pd.read_csv(DATA_PATH, dtype=str)

            def clean_number(value):
                if pd.isna(value):
                    return float("nan")
                number = parse_number(str(value))      # same rules as my own code
                return float("nan") if number is None else number

            df["qty_clean"] = df["quantity"].apply(clean_number)
            df["price_clean"] = df["unit_price_kes"].apply(clean_number)

            df_clean = df.dropna(subset=["qty_clean", "price_clean"]).copy()
            df_clean["revenue"] = df_clean["qty_clean"] * df_clean["price_clean"]

            pd_total_rev = float(df_clean["revenue"].sum())
            pd_prod_rev = df_clean.groupby("product")["revenue"].sum().to_dict()
            pd_top_3 = sorted(pd_prod_rev.items(), key=lambda item: (-item[1], item[0]))[:3]
            pd_mean = float(df_clean["revenue"].mean())
            pd_median = float(df_clean["revenue"].median())
            pd_min = float(df_clean["revenue"].min())
            pd_max = float(df_clean["revenue"].max())

            def close(a, b):
                return math.isclose(a, b, abs_tol=0.01)

            total_ok = close(total_rev, pd_total_rev)
            prod_ok = set(product_totals) == set(pd_prod_rev) and all(
                close(product_totals[k], pd_prod_rev[k]) for k in product_totals
            )
            top3_ok = len(top_three) == len(pd_top_3) and all(
                mine[0] == theirs[0] and close(mine[1], theirs[1])
                for mine, theirs in zip(top_three, pd_top_3)
            )
            stats_ok = close(rev_mean, pd_mean) and close(rev_median, pd_median)
            minmax_ok = close(rev_min, pd_min) and close(rev_max, pd_max)

            print("=== pandas verification ===")
            if total_ok:
                print("Total revenue      : MATCH")
            else:
                print(f"Total revenue      : MISMATCH (mine {total_rev:,.2f}, pandas {pd_total_rev:,.2f})")
            print(f"Revenue by product : {'MATCH' if prod_ok else 'MISMATCH'}")
            print(f"Top 3              : {'MATCH' if top3_ok else 'MISMATCH'}")
            print(f"Mean / median      : {'MATCH' if stats_ok else 'MISMATCH'}")
            print(f"Min / max          : {'MATCH' if minmax_ok else 'MISMATCH'}")

    report_text = buffer.getvalue()
    print(report_text)

    OUTPUT_PATH.parent.mkdir(exist_ok=True)
    OUTPUT_PATH.write_text(report_text, encoding="utf-8")
    print(f"Report saved to {OUTPUT_PATH}")


main()

Rows read: 50
Columns found: sale_id, sale_date, store, product, quantity, unit_price_kes, payment_method
=== Sales report: September 2026 ===
Rows read          : 50
Rows used          : 47
Rows skipped       : 3   (S-3030: Invalid unit price, S-3035: Invalid quantity, S-3039: Invalid unit price)

Total revenue (KES): 1,317,210.00

Revenue by product (KES)
  HP 15 Laptop              389,994.00
  Hisense 43in Smart TV     272,993.00
  Lenovo Tab M10            199,992.00
  Tecno Spark 20            161,988.00
  Infinix Hot 40            63,996.00
  Samsung Galaxy A15        56,997.00
  JBL Go 3 Speaker          49,491.00
  Anker Power Bank 20000    38,691.00
  Xiaomi Redmi Buds         31,990.00
  Oraimo FreePods           19,593.00
  USB-C Charger 25W         19,487.00
  Safaricom Neon Ray        11,998.00

Top 3 products
  1. HP 15 Laptop - KES 389,994.00
  2. Hisense 43in Smart TV - KES 272,993.00
  3. Lenovo Tab M10 - KES 199,992.00

High-value transactions (>= KES 50,000.00): 5
 